# Chapter 1: Pallet Town -- What Is Causal Inference?

---

> *"Every trainer starts their journey in Pallet Town. Every causal analyst starts by learning to distinguish correlation from causation."*

In this chapter, you will learn:

1. Why **correlation does not imply causation** -- and what traps even smart trainers fall into
2. The **Fundamental Problem of Causal Inference** -- we can never observe the road not taken
3. The **Potential Outcomes Framework** (Rubin Causal Model) -- ATE, ATT, ATC, and selection bias
4. How **confounders** distort naive treatment-effect estimates
5. How to think in **counterfactuals** -- the foundation of everything that follows

---

## 1. Setup & Data Loading

In [ ]:
# Core scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Kanto utilities
from kanto_utils import (
    load_trainers, apply_kanto_theme, type_color, type_colors_dict,
    pokemon_scatter, difference_in_means, balance_table,
    oak_says, blue_says, blues_mistake, badge_earned,
    confounder_slider, starter_selector,
)

# Apply the Kanto visual theme to all plots
apply_kanto_theme()

# Reproducibility
np.random.seed(151)  # 151 original Pokemon

# Load the Kanto trainers dataset
trainers = load_trainers()
print(f"Loaded {len(trainers):,} trainers from the Kanto region.")
trainers.head()

In [ ]:
oak_says(
    "Welcome to Pallet Town, trainer! I'm Professor Oak. "
    "Before you set off on your Pokemon journey, you need to learn "
    "the most important lesson in all of research: <b>correlation is not causation</b>. "
    "Many trainers -- even my grandson Blue -- make this mistake. "
    "Today, we'll build the foundations that let you tell the difference."
)

---
## 2. Correlation vs. Causation

Let's start with a question Blue might ask:

> *"I picked Squirtle and earned 6 badges. My rival picked Charmander and only has 3. Clearly Water starters **cause** more badges!"*

Let's see if the data supports this claim.

In [ ]:
# Average badges by starter type
badges_by_starter = trainers.groupby('starter_type')['badges'].agg(['mean', 'std', 'count'])
badges_by_starter.columns = ['Mean Badges', 'Std Dev', 'N Trainers']
badges_by_starter = badges_by_starter.round(2)
print("Badges earned by starter type:")
badges_by_starter

In [ ]:
# Visualise the apparent relationship
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: box plot of badges by starter type
starter_types = ['Fire', 'Water', 'Grass']
colors = [type_color(t) for t in starter_types]

bp_data = [trainers[trainers['starter_type'] == t]['badges'] for t in starter_types]
bp = axes[0].boxplot(bp_data, labels=starter_types, patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[0].set_ylabel('Badges Earned')
axes[0].set_xlabel('Starter Type')
axes[0].set_title('Badges by Starter Type')

# Right: scatter of starter_type vs badges with jitter
for i, t in enumerate(starter_types):
    subset = trainers[trainers['starter_type'] == t]
    jitter = np.random.normal(0, 0.08, size=len(subset))
    axes[1].scatter(
        np.full(len(subset), i) + jitter,
        subset['badges'],
        alpha=0.3, s=20, color=type_color(t), label=t
    )
axes[1].set_xticks(range(len(starter_types)))
axes[1].set_xticklabels(starter_types)
axes[1].set_ylabel('Badges Earned')
axes[1].set_xlabel('Starter Type')
axes[1].set_title('Individual Trainers (jittered)')
axes[1].legend(frameon=True)

fig.suptitle('Does Starter Type Cause More Badges?', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
blue_says(
    "See? I told you -- Water starters are the best! Look at those averages. "
    "If you want more badges, just pick Squirtle. Case closed."
)

### Revealing the Confounders

Not so fast, Blue. Let's check whether the **types of trainers** who choose each starter differ systematically. If wealthier or more experienced trainers tend to pick certain starters, then the *starter choice* is **confounded** with trainer quality.

In [ ]:
# Check confounders: trainer_experience and wealth by starter type
confounders = trainers.groupby('starter_type')[['trainer_experience', 'wealth', 'strategy_score', 'dedication']].mean().round(2)
print("Average trainer characteristics by starter type:")
confounders

In [ ]:
# Visualise the confounding
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, var, title in zip(
    axes,
    ['trainer_experience', 'wealth', 'strategy_score'],
    ['Trainer Experience', 'Wealth Level', 'Strategy Score']
):
    for t in starter_types:
        subset = trainers[trainers['starter_type'] == t]
        ax.hist(subset[var], bins=20, alpha=0.5, color=type_color(t), label=t, density=True)
    ax.set_xlabel(title)
    ax.set_ylabel('Density')
    ax.set_title(f'{title} by Starter Type')
    ax.legend(frameon=True, fontsize=9)

fig.suptitle('Confounders: Trainer Characteristics Differ by Starter Choice',
             fontsize=14, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
blues_mistake(
    claim="Water starters cause more badges because the average is higher.",
    reality=(
        "Trainers who choose Water starters tend to have more experience and "
        "higher strategy scores. These pre-existing differences -- not the starter "
        "itself -- explain much of the badge gap. This is <b>confounding</b>."
    )
)

---
## 3. The Fundamental Problem of Causal Inference

To ask a causal question properly, we need the **potential outcomes** framework.

For each trainer $i$, define:
- $Y_i(1)$ = outcome (badges) if trainer $i$ **uses** the Exp. Share item
- $Y_i(0)$ = outcome (badges) if trainer $i$ **does not use** the Exp. Share

The **individual treatment effect** is: $\tau_i = Y_i(1) - Y_i(0)$

**The Fundamental Problem**: For each trainer, we observe *only one* of $Y_i(1)$ or $Y_i(0)$, never both. The unobserved outcome is the **counterfactual**.

In [ ]:
oak_says(
    "The Fundamental Problem of Causal Inference (Holland, 1986) is simple but profound: "
    "<b>we can never observe the same unit in both the treated and untreated state at the same time.</b> "
    "It's like asking: 'What would have happened if Red had picked Bulbasaur instead of Charmander?' "
    "He can't do both. The road not taken remains forever unknown."
)

In [ ]:
# Create a potential outcomes table for 8 trainers
# We'll use the Exp. Share as our treatment
sample = trainers.sample(8, random_state=151).reset_index(drop=True)

# Observed outcomes
po_table = pd.DataFrame({
    'Trainer': sample['trainer_name'],
    'Treatment (D)': sample['exp_share_used'],
    'Observed Y': sample['badges'],
})

# The "true" potential outcomes -- Y(1) and Y(0)
# For the observed state, Y(D) = Observed Y
# For the counterfactual, we mark it as unknown
po_table['Y(1)'] = po_table.apply(
    lambda row: str(int(row['Observed Y'])) if row['Treatment (D)'] == 1 else '?', axis=1
)
po_table['Y(0)'] = po_table.apply(
    lambda row: str(int(row['Observed Y'])) if row['Treatment (D)'] == 0 else '?', axis=1
)
po_table['Individual TE'] = '?'  # We can never compute this for real data

print("Potential Outcomes Table (Exp. Share Treatment)")
print("=" * 70)
print("D = 1 means the trainer used Exp. Share; D = 0 means they did not.")
print("'?' marks the counterfactual outcome we CANNOT observe.\n")
po_table[['Trainer', 'Treatment (D)', 'Y(1)', 'Y(0)', 'Individual TE']]

In [ ]:
oak_says(
    "Notice the column of question marks. For treated trainers, we see $Y(1)$ "
    "but not $Y(0)$. For untreated trainers, we see $Y(0)$ but not $Y(1)$. "
    "The individual treatment effect $\\tau_i = Y_i(1) - Y_i(0)$ is <b>never directly observable</b>. "
    "This is why causal inference is fundamentally a <b>missing data problem</b>."
)

#### What Can and Cannot Be Computed

In [ ]:
treated_obs = sample[sample['exp_share_used'] == 1]['badges']
control_obs = sample[sample['exp_share_used'] == 0]['badges']

print("What we CAN compute from observed data:")
print(f"  E[Y | D=1] = {treated_obs.mean():.2f}  (mean badges among treated)")
print(f"  E[Y | D=0] = {control_obs.mean():.2f}  (mean badges among untreated)")
print(f"  Naive difference = {treated_obs.mean() - control_obs.mean():.2f}")
print()
print("What we CANNOT compute:")
print("  E[Y(1) - Y(0)]  -- the true Average Treatment Effect")
print("  Any individual treatment effect tau_i")
print()
print("The naive difference equals the ATE ONLY if treatment is independent")
print("of potential outcomes (i.e., there is no selection bias).")

---
## 4. Potential Outcomes Framework: ATE, ATT, ATC, and Selection Bias

Since individual treatment effects are unobservable, we work with **population averages**.

| Estimand | Definition | Interpretation |
|----------|-----------|----------------|
| **ATE** | $E[Y(1) - Y(0)]$ | Average effect for a randomly chosen trainer |
| **ATT** | $E[Y(1) - Y(0) \mid D=1]$ | Average effect for trainers who *actually used* Exp. Share |
| **ATC** | $E[Y(1) - Y(0) \mid D=0]$ | Average effect for trainers who *did not use* it |

The **Selection Bias Decomposition** tells us:

$$\underbrace{E[Y|D=1] - E[Y|D=0]}_{\text{Naive difference}} = \underbrace{\text{ATT}}_{\text{Causal effect on treated}} + \underbrace{E[Y(0)|D=1] - E[Y(0)|D=0]}_{\text{Selection bias}}$$

In [ ]:
oak_says(
    "Here is the key insight. The <b>naive difference in means</b> between treated and "
    "untreated groups conflates two things: (1) the genuine causal effect of treatment, "
    "and (2) pre-existing differences between the groups -- <b>selection bias</b>. "
    "If stronger trainers are more likely to use Exp. Share, then the naive estimate "
    "will be biased upward even if Exp. Share has zero causal effect."
)

In [ ]:
# Teaching trick: simulate the FULL potential outcomes schedule
# so we can compute ATE, ATT, ATC exactly and show selection bias.
#
# DGP: Y(0) depends on trainer_experience and strategy_score.
#       Y(1) = Y(0) + tau_i, where tau_i ~ N(1.0, 0.3).
#       Treatment assignment depends on experience (selection on observables).

rng = np.random.default_rng(151)

n = len(trainers)

# Potential outcome under control: driven by experience and strategy
Y0 = (
    0.3 * trainers['trainer_experience']
    + 0.05 * trainers['strategy_score']
    + 0.2 * trainers['wealth']
    + rng.normal(0, 0.5, n)
).values

# True individual treatment effects -- Exp. Share has a real effect of ~1.0 badge
tau_i = 1.0 + 0.3 * rng.standard_normal(n)

# Potential outcome under treatment
Y1 = Y0 + tau_i

# Observed treatment (selection: experienced trainers more likely to use Exp. Share)
D = trainers['exp_share_used'].values

# Observed outcome
Y_obs = D * Y1 + (1 - D) * Y0

print("=" * 60)
print("TEACHING MODE: Full Potential Outcomes (God's-eye view)")
print("=" * 60)
print(f"\nTrue ATE  = E[Y(1) - Y(0)]         = {np.mean(tau_i):.4f}")
print(f"True ATT  = E[Y(1) - Y(0) | D=1]    = {np.mean(tau_i[D == 1]):.4f}")
print(f"True ATC  = E[Y(1) - Y(0) | D=0]    = {np.mean(tau_i[D == 0]):.4f}")
print()

# Naive difference
naive_diff = Y_obs[D == 1].mean() - Y_obs[D == 0].mean()
print(f"Naive difference E[Y|D=1] - E[Y|D=0] = {naive_diff:.4f}")

# Selection bias decomposition
att = np.mean(Y1[D == 1] - Y0[D == 1])
selection_bias = np.mean(Y0[D == 1]) - np.mean(Y0[D == 0])

print(f"\nSelection Bias Decomposition:")
print(f"  Naive difference = ATT + Selection Bias")
print(f"  {naive_diff:.4f}          = {att:.4f} + {selection_bias:.4f}")
print(f"  Check: {att + selection_bias:.4f} (should match naive diff)")
print(f"\n  Selection bias = E[Y(0)|D=1] - E[Y(0)|D=0]")
print(f"  = {np.mean(Y0[D == 1]):.4f} - {np.mean(Y0[D == 0]):.4f} = {selection_bias:.4f}")

if selection_bias > 0:
    print(f"\n  Positive selection bias: trainers who use Exp. Share would")
    print(f"  have earned MORE badges even WITHOUT it. The naive estimate")
    print(f"  overstates the true causal effect.")
else:
    print(f"\n  Negative selection bias: trainers who use Exp. Share would")
    print(f"  have earned FEWER badges without it. The naive estimate")
    print(f"  understates the true causal effect.")

In [ ]:
# Visualise the decomposition
fig, ax = plt.subplots(figsize=(8, 5))

bar_labels = ['Naive Difference', 'True ATT', 'Selection Bias']
bar_values = [naive_diff, att, selection_bias]
bar_colors = ['#EE1515', '#3B4CCA', '#FFD733']

bars = ax.bar(bar_labels, bar_values, color=bar_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel('Effect Size (badges)')
ax.set_title('Selection Bias Decomposition\nNaive Difference = ATT + Selection Bias')
ax.axhline(0, color='grey', linewidth=0.8, linestyle=':')
plt.tight_layout()
plt.show()

---
## 5. The Confounder Slider Widget

Let's see **interactively** how controlling for a confounder changes our estimate. Use the slider to restrict the sample to trainers within a narrow range of `trainer_experience`. As you narrow the range, you are comparing more similar trainers -- and the estimated treatment effect moves closer to the truth.

In [ ]:
oak_says(
    "Try narrowing the slider to compare only trainers with <b>similar experience levels</b>. "
    "When you hold the confounder constant, the naive estimate changes -- "
    "often getting smaller. That's because you are removing selection bias "
    "by conditioning on the confounder. This is the intuition behind "
    "<b>selection on observables</b> strategies we'll study later."
)

In [ ]:
confounder_slider(
    data=trainers,
    treatment='exp_share_used',
    outcome='badges',
    confounder='trainer_experience'
)

---
## 6. Counterfactual Thinking

Let's practice thinking about counterfactuals using three specific trainers who chose Charmander. For each trainer, we observe their real outcomes. The question is: **What would have happened if they had chosen Squirtle instead?**

In [ ]:
# Select 3 Charmander trainers with different profiles
charmanders = trainers[trainers['starter_species'] == 'Charmander'].copy()

# Pick trainers with low, medium, and high experience
low_exp = charmanders.nsmallest(1, 'trainer_experience').iloc[0]
mid_idx = (charmanders['trainer_experience'] - charmanders['trainer_experience'].median()).abs().idxmin()
mid_exp = charmanders.loc[mid_idx]
high_exp = charmanders.nlargest(1, 'trainer_experience').iloc[0]

counterfactual_trainers = pd.DataFrame([low_exp, mid_exp, high_exp])

print("Three Charmander Trainers:")
print("=" * 70)
display_cols = ['trainer_name', 'starter_species', 'trainer_experience',
                'strategy_score', 'wealth', 'badges', 'total_battles_won']
counterfactual_trainers[display_cols]

In [ ]:
# Counterfactual reasoning exercise
print("Counterfactual Thinking Exercise")
print("=" * 70)
print()

for _, trainer in counterfactual_trainers.iterrows():
    print(f"Trainer: {trainer['trainer_name']}")
    print(f"  Starter: {trainer['starter_species']} (Fire type)")
    print(f"  Experience: {trainer['trainer_experience']:.1f} | Strategy: {trainer['strategy_score']:.1f}")
    print(f"  OBSERVED outcome: {int(trainer['badges'])} badges earned")
    print(f"  COUNTERFACTUAL: What if they had chosen Squirtle (Water type)?")
    print(f"    -> We can NEVER observe this. That's the Fundamental Problem.")
    print(f"    -> But we can look at SIMILAR trainers who DID pick Squirtle...")
    
    # Find a similar trainer who chose Squirtle
    squirtles = trainers[trainers['starter_species'] == 'Squirtle']
    exp_diff = (squirtles['trainer_experience'] - trainer['trainer_experience']).abs()
    closest = squirtles.loc[exp_diff.idxmin()]
    print(f"    -> Closest Squirtle trainer: {closest['trainer_name']} "
          f"(exp={closest['trainer_experience']:.1f}) earned {int(closest['badges'])} badges")
    print()

oak_says(
    "This matching approach -- finding similar units who received different treatments -- is "
    "one of the oldest ideas in causal inference. But 'similar' on one variable might not be "
    "similar on all the variables that matter. We'll develop more rigorous approaches "
    "in the chapters ahead."
)

---
## Challenge Exercises

Now it's your turn! Complete the three challenges below to test your understanding.

### Challenge 1: Compute ATE and Selection Bias for a New Subset

Using only trainers from **Pewter City** and **Cerulean City**, compute:
1. The naive difference in `total_battles_won` between trainers who used Exp. Share ($D=1$) and those who did not ($D=0$)
2. Using the simulated potential outcomes from Section 4, compute the ATT and selection bias for this subset

**Hint**: Filter the data on `hometown`, then use the `Y0`, `Y1`, `D` arrays (indexed to the same rows).

In [ ]:
# CHALLENGE 1: Your code here
# ---------------------------

# Step 1: Filter to Pewter City and Cerulean City trainers
subset_mask = trainers['hometown'].isin(['Pewter City', 'Cerulean City'])
subset = trainers[subset_mask].copy()
print(f"Subset size: {len(subset)} trainers")

# Step 2: Naive difference in total_battles_won
naive = (subset[subset['exp_share_used'] == 1]['total_battles_won'].mean() -
         subset[subset['exp_share_used'] == 0]['total_battles_won'].mean())
print(f"\nNaive difference in total_battles_won: {naive:.3f}")

# Step 3: Use the simulated potential outcomes for the ATT/selection bias
D_sub = D[subset_mask]
Y0_sub = Y0[subset_mask]
Y1_sub = Y1[subset_mask]

att_sub = np.mean(Y1_sub[D_sub == 1] - Y0_sub[D_sub == 1])
sel_bias_sub = np.mean(Y0_sub[D_sub == 1]) - np.mean(Y0_sub[D_sub == 0])

print(f"\nTrue ATT (simulated):       {att_sub:.4f}")
print(f"Selection Bias (simulated): {sel_bias_sub:.4f}")
print(f"Sum (should ~ naive):       {att_sub + sel_bias_sub:.4f}")

### Challenge 2: Identify the Confounder

Consider the relationship between `cave_training` (treatment: did the trainer do extra training in caves?) and `badges` (outcome).

1. Compute the naive difference in badges between cave trainers and non-cave trainers.
2. Find at least one variable that is correlated with *both* `cave_training` and `badges`. This is a potential confounder.
3. Explain *why* this confounder biases the naive estimate.

**Hint**: Check `dedication`, `trainer_experience`, or `patience`.

In [ ]:
# CHALLENGE 2: Your code here
# ---------------------------

# Step 1: Naive difference
cave_yes = trainers[trainers['cave_training'] == 1]['badges']
cave_no  = trainers[trainers['cave_training'] == 0]['badges']
naive_cave = cave_yes.mean() - cave_no.mean()
print(f"Naive difference (cave training -> badges): {naive_cave:.3f}")

# Step 2: Check potential confounders
candidates = ['dedication', 'trainer_experience', 'patience', 'natural_talent']
print("\nCorrelations with cave_training and badges:")
for c in candidates:
    r_treat = trainers[c].corr(trainers['cave_training'])
    r_outcome = trainers[c].corr(trainers['badges'])
    flag = " <-- CONFOUNDER" if abs(r_treat) > 0.05 and abs(r_outcome) > 0.05 else ""
    print(f"  {c:25s}  r(cave_training)={r_treat:+.3f}  r(badges)={r_outcome:+.3f}{flag}")

# Step 3: Explanation
print("\nExplanation:")
print("Variables correlated with BOTH the treatment and the outcome are confounders.")
print("For example, if 'dedication' predicts both cave_training and badges,")
print("then the naive difference captures the effect of dedication (not just caves).")

### Challenge 3: Simulate Potential Outcomes and Estimate Bias

Write a function `simulate_and_estimate(n, true_ate, confounding_strength, seed)` that:

1. Generates a confounder $X \sim N(0, 1)$
2. Creates potential outcomes: $Y(0) = 2 + X + \epsilon$, $Y(1) = Y(0) + \text{true\_ate}$
3. Assigns treatment with confounding: $P(D=1|X) = \text{logistic}(\text{confounding\_strength} \times X)$
4. Computes the naive estimate and the bias

Run it for `confounding_strength` values of 0, 0.5, 1.0, and 2.0. Plot how bias changes.

In [ ]:
# CHALLENGE 3: Your code here
# ---------------------------

from scipy.special import expit  # logistic function

def simulate_and_estimate(n=2000, true_ate=1.0, confounding_strength=0.0, seed=151):
    """Simulate potential outcomes with confounding and return naive estimate + bias."""
    rng = np.random.default_rng(seed)
    
    # 1. Confounder
    X = rng.standard_normal(n)
    
    # 2. Potential outcomes
    epsilon = rng.normal(0, 0.5, n)
    Y0 = 2 + X + epsilon
    Y1 = Y0 + true_ate
    
    # 3. Treatment with confounding
    prob_treat = expit(confounding_strength * X)
    D = rng.binomial(1, prob_treat)
    
    # 4. Observed outcome and naive estimate
    Y_obs = D * Y1 + (1 - D) * Y0
    naive = Y_obs[D == 1].mean() - Y_obs[D == 0].mean()
    bias = naive - true_ate
    
    return {'naive_estimate': naive, 'true_ate': true_ate, 'bias': bias}

# Run for different confounding strengths
strengths = np.linspace(0, 2.0, 20)
biases = [simulate_and_estimate(confounding_strength=s)['bias'] for s in strengths]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(strengths, biases, 'o-', color='#EE1515', linewidth=2, markersize=6)
ax.axhline(0, color='grey', linestyle=':', linewidth=1)
ax.set_xlabel('Confounding Strength')
ax.set_ylabel('Bias (Naive Estimate - True ATE)')
ax.set_title('How Confounding Strength Affects Selection Bias')
plt.tight_layout()
plt.show()

print("Key insight: As confounding gets stronger, the naive estimate diverges")
print("further from the truth. With zero confounding (random assignment),")
print("the bias is approximately zero.")

---
## Chapter Summary

In this chapter, you learned:

1. **Correlation is not causation** -- differences between groups may reflect pre-existing characteristics, not causal effects.
2. **The Fundamental Problem of Causal Inference** -- we can only observe one potential outcome for each unit.
3. **Potential outcomes** $Y(1)$ and $Y(0)$ let us formally define causal effects: ATE, ATT, ATC.
4. **Selection bias** arises when treatment assignment is related to potential outcomes.
5. **Confounders** are variables that affect both treatment and outcome, creating spurious associations.
6. **Counterfactual thinking** is the foundation: "What would have happened under the alternative treatment?"

In the next chapter, we travel to **Pewter City** to learn how **randomized experiments** solve the selection bias problem -- and earn our first badge!

In [ ]:
badge_earned("Boulder Badge Preview", 1)